# What is Changed:

## Architecture
1. Replace 5x5 Conv2D by MobileNet version 1 block with Kernel=3 and stride=1
2. Add BatchNormalization after every Conv2D

## Parameters
1. weight decay: 5e-4 to 1e-4
2. initial LearningRate: 1e-2 to 1e-3
3. LearningRate reduce factor: 0.1 to 0.5
4. minimum LearningRate: 1e-5 to 1e-6


# Network In Network Architecture

## Using NIN as a Backbone for Object Detection
Yes, you can use those three mlpconv blocks as a feature extraction backbone. Here is how that translates to Object Detection:

### Classification (Your current setup):
You average the entire map (GAP) to get a single label for the whole image.

### Detection/Localization:
Instead of averaging, you look for the peak activations in those 10 maps. The $(x, y)$ position of the highest activation in the "Car" map tells you where the car is located in the $32 \times 32$ grid.3.

### Why this is powerful for CIFAR-10
Even though CIFAR-10 images are tiny ($32 \times 32$), objects aren't always perfectly centered.By using the NIN backbone, you retain spatial awareness.`If you were to replace the GAP layer with a Region Proposal Network (RPN) or a Heatmap Regression head, you could theoretically draw a bounding box around the digit or object.`

## One "Peer-to-Peer" Correction
While the authors say this motivates object detection, keep in mind that CIFAR-10 only has one primary object per image. `To do "real" object detection (multiple objects, different sizes), you would typically use this NIN backbone inside a framework like SSD (Single Shot Detector) or YOLO, where the backbone extracts the features and specialized "heads" predict the bounding box coordinates ($x, y, w, h$).` By using all three mlpconv layers, you are capturing everything from low-level edges to the high-level "confidence" the authors mentioned. `It’s a very efficient backbone because it lacks the heavy parameter count of fully connected layers!`


# Is there any follow-up research e.g author add shortcut connection after each mlpConv layer or replace Standard CNN(kernel=5,5) of each mlpConv layer with DepthWiseConv2D and PointWiseConv2D (like MobileNet) etc

You’ve hit on the exact evolution that happened in deep learning between 2014 and 2017! The "Network In Network" (NIN) design was the "missing link" between early CNNs and the modern, efficient architectures we use today.

While the original authors didn't release a "NIN v2" with those specific changes, the rest of the research community took the NIN concepts and evolved them into three major directions:

## Shortcuts and NIN:
The Birth of ResNetThe idea of adding shortcut (skip) connections after layers—including NIN-style blocks—was the breakthrough of `Kaiming He et al. (2015) in the ResNet paper`.

1. The Connection:
ResNet actually uses "Bottleneck Blocks" that look remarkably like NIN. They use a $1 \times 1$ convolution to reduce dimensions, a $3 \times 3$ for spatial features, and another $1 \times 1$ to expand them back, all wrapped in a shortcut connection.

2. `Research Paper: "Deep Residual Learning for Image Recognition" (2015).`

## Depthwise + Pointwise: The MobileNet Evolution
Your idea to replace the $5 \times 5$ or $3 \times 3$ standard convolutions with Depthwise Separable Convolutions is exactly what Google researchers did to create MobileNet.
1. How it relates to NIN:
MobileNet is essentially a series of "Micro-networks." It takes the $1 \times 1$ convolution (the heart of NIN) and pairs it with a Depthwise convolution to make the "Learnable Weighted Average" incredibly cheap in terms of computation.

2. Research Paper: `"MobileNets: Efficient Convolutional Neural Networks for Mobile Vision Applications" (Howard et al., 2017).`

## The "Inception" Module (NIN on Steroids)
Shortly after NIN, the GoogLeNet team acknowledged NIN as their primary inspiration. They took the $1 \times 1$ convolution and used it to create the Inception Module.

1. The Improvement:
Instead of one $5 \times 5$ layer, they ran $1 \times 1, 3 \times 3,$ and $5 \times 5$ convolutions in parallel and concatenated them. This is like having three different "NIN" blocks looking at the same patch of image at different scales simultaneously.

2. Research Paper: `"Going Deeper with Convolutions" (Szegedy et al., 2014).`

## Modern "NIN-like" Backbones for Detection
If you want to use a NIN-style backbone for object detection today, you are essentially looking at SqueezeNet.

1. SqueezeNet (2016):
This paper is the direct spiritual successor to NIN. The authors explicitly stated their goal was to create a model with NIN-level accuracy but with 50x fewer parameters.

2. The Architecture:
It uses "Fire Modules" which consist of a "Squeeze" layer ($1 \times 1$ conv) and an "Expand" layer (mix of $1 \times 1$ and $3 \times 3$ convs). Like NIN, it ends with Global Average Pooling.

3. Detection Use Case:
SqueezeNet is a very popular backbone for SqueezeDet, a small, fast object detector.

# Imports And Download Dataset

In [ ]:
import os
import json
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras import layers, models, callbacks, regularizers, optimizers
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             classification_report, accuracy_score)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.calibration import calibration_curve
from scipy.optimize import minimize
from scipy.stats import entropy


# 1. Load the dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(f"Training shape: {x_train.shape}") # Should be (50000, 32, 32, 3)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Training shape: (50000, 32, 32, 3)


# Save And Create Test/Train DIR
So that keras can load data from directory with custom data loader `get_keras_dataset()`

In [ ]:
# Define base directory for CIFAR-10 files
CIFAR10_BASE_DIR = "/kaggle/working/cifar10_files"
CIFAR10_TRAIN_DIR = os.path.join(CIFAR10_BASE_DIR, "train")
CIFAR10_TEST_DIR = os.path.join(CIFAR10_BASE_DIR, "test")

# Create main directories
os.makedirs(CIFAR10_TRAIN_DIR, exist_ok=True)
os.makedirs(CIFAR10_TEST_DIR, exist_ok=True)

# CIFAR-10 has 10 classes
num_classes_cifar10 = 10

# Create class subdirectories for train and test
for i in range(num_classes_cifar10):
    os.makedirs(os.path.join(CIFAR10_TRAIN_DIR, str(i)), exist_ok=True)
    os.makedirs(os.path.join(CIFAR10_TEST_DIR, str(i)), exist_ok=True)

print(f"Directories created: {CIFAR10_TRAIN_DIR} and {CIFAR10_TEST_DIR} with class subfolders.")

# Save training images
print("Saving training images...")
for i in range(x_train.shape[0]):
    # x_train is already uint8 [0, 255]
    img = x_train[i]
    # y_train is integer label (N, 1), so take the first element
    label = y_train[i][0]
    # Define filename
    filename = os.path.join(CIFAR10_TRAIN_DIR, str(label), f"image_{i:05d}.png")
    cv2.imwrite(filename, img)
print(f"Saved {x_train.shape[0]} training images to {CIFAR10_TRAIN_DIR}.")

# Save testing images
print("Saving testing images...")
for i in range(x_test.shape[0]):
    # x_test is already uint8 [0, 255]
    img = x_test[i]
    # y_test is integer label (N, 1), so take the first element
    label = y_test[i][0]
    # Define filename
    filename = os.path.join(CIFAR10_TEST_DIR, str(label), f"image_{i:05d}.png")
    cv2.imwrite(filename, img)
print(f"Saved {x_test.shape[0]} testing images to {CIFAR10_TEST_DIR}.")


Directories created: /kaggle/working/cifar10_files/train and /kaggle/working/cifar10_files/test with class subfolders.
Saving training images...
Saved 50000 training images to /kaggle/working/cifar10_files/train.
Saving testing images...
Saved 10000 testing images to /kaggle/working/cifar10_files/test.


In [ ]:
import os
import tensorflow as tf
import numpy as np
import cv2 # For saving images
import random # For random augmentation values
from tensorflow import keras
from tensorflow.keras import layers, models


num_classes = 10
IMG_SIZE = (32, 32)
BATCH_SIZE = 64


# Define augmented directories
AUGMENTED_BASE_DIR = "/kaggle/working/augmented_cifar10"
AUGMENTED_TRAIN_DIR = os.path.join(AUGMENTED_BASE_DIR, "train")
AUGMENTED_TEST_DIR = os.path.join(AUGMENTED_BASE_DIR, "test")

def create_and_save_augmented_images_to_disk(
    source_dir,
    target_dir):
    """
    Generates augmented images from a directory of raw images and saves them to disk.
    Applies custom augmentations (rotation, translation, zoom) using OpenCV and NumPy.
    Saves one image for each augmentation type (rotation, translation, zoom) per original image.
    """

    print(f"--- Creating and Saving Augmented Images to {target_dir} from {source_dir} ---")

    # Use global num_classes, IMG_SIZE, BATCH_SIZE
    global num_classes, IMG_SIZE, BATCH_SIZE

    # Create class subdirectories
    for i in range(num_classes):
        os.makedirs(os.path.join(target_dir, str(i)), exist_ok=True)

    # Create dataset from directory for augmentation
    # Images are loaded as float32 in [0, 255] by default
    dataset_to_augment = tf.keras.utils.image_dataset_from_directory(
        directory=source_dir,
        labels="inferred",
        label_mode="categorical",
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False # We're iterating to save, so shuffle is not needed.
    )

    total_saved_images = 0
    original_image_counter = 0

    for batch_idx, (images_batch, labels_batch) in enumerate(dataset_to_augment):
        for original_img_idx_in_batch in range(images_batch.shape[0]):
            original_image_tensor = images_batch[original_img_idx_in_batch]
            original_image_np = original_image_tensor.numpy() # Image is float32 [0, 255]
            original_label_one_hot = labels_batch[original_img_idx_in_batch].numpy()
            original_label = np.argmax(original_label_one_hot)

            img_to_augment = original_image_np.astype(np.uint8)

            # --- Apply Rotation ---
            angle = random.uniform(-15, 15)
            M_rot = cv2.getRotationMatrix2D((IMG_SIZE[0] // 2, IMG_SIZE[1] // 2), angle, 1)
            rotated_image = cv2.warpAffine(img_to_augment, M_rot, IMG_SIZE, borderMode=cv2.BORDER_REFLECT_101)
            filename_rot = os.path.join(target_dir, str(original_label),
                                    f"orig_{original_image_counter:05d}_rot.png")
            cv2.imwrite(filename_rot, rotated_image.astype(np.uint8))
            total_saved_images += 1

            # --- Apply Translation ---
            tx = random.uniform(-1, 1)
            ty = random.uniform(-1, 1)
            M_trans = np.float32([[1, 0, tx], [0, 1, ty]])
            translated_image = cv2.warpAffine(img_to_augment, M_trans, IMG_SIZE, borderMode=cv2.BORDER_REFLECT_101)
            filename_trans = os.path.join(target_dir, str(original_label),
                                        f"orig_{original_image_counter:05d}_trans.png")
            cv2.imwrite(filename_trans, translated_image.astype(np.uint8))
            total_saved_images += 1

            # --- Apply Zoom ---
            zoom_factor = random.uniform(0.98, 1.02)
            M_zoom = cv2.getRotationMatrix2D((IMG_SIZE[0] // 2, IMG_SIZE[1] // 2), 0, zoom_factor)
            zoomed_image = cv2.warpAffine(img_to_augment, M_zoom, IMG_SIZE, borderMode=cv2.BORDER_REFLECT_101)
            filename_zoom = os.path.join(target_dir, str(original_label),
                                        f"orig_{original_image_counter:05d}_zoom.png")
            cv2.imwrite(filename_zoom, zoomed_image.astype(np.uint8))
            total_saved_images += 1

            original_image_counter += 1

    print(f"Successfully saved {total_saved_images} augmented images to {target_dir}.")
    return total_saved_images

# Ensure base augmented directory exists
os.makedirs(AUGMENTED_BASE_DIR, exist_ok=True)

print("Starting augmentation process...")
# Augment and save training images
saved_train_count = create_and_save_augmented_images_to_disk(
    source_dir=CIFAR10_TRAIN_DIR, # Use the global CIFAR10_TRAIN_DIR
    target_dir=AUGMENTED_TRAIN_DIR
    # No augmentation_pipeline_model is passed here
)

# Augment and save testing images
saved_test_count = create_and_save_augmented_images_to_disk(
    source_dir=CIFAR10_TEST_DIR, # Use the global CIFAR10_TEST_DIR
    target_dir=AUGMENTED_TEST_DIR
    # No augmentation_pipeline_model is passed here
)

print(f"Total augmented images saved across train and test sets: {saved_train_count + saved_test_count}")
print("Augmentation process complete.")

Starting augmentation process...
--- Creating and Saving Augmented Images to /kaggle/working/augmented_cifar10/train from /kaggle/working/cifar10_files/train ---
Found 50000 files belonging to 10 classes.
Successfully saved 150000 augmented images to /kaggle/working/augmented_cifar10/train.
--- Creating and Saving Augmented Images to /kaggle/working/augmented_cifar10/test from /kaggle/working/cifar10_files/test ---
Found 10000 files belonging to 10 classes.
Successfully saved 30000 augmented images to /kaggle/working/augmented_cifar10/test.
Total augmented images saved across train and test sets: 180000
Augmentation process complete.


In [ ]:
# Zip and trigger download
!zip -r /kaggle/working/augmented_cifar10/train.zip  /kaggle/working/augmented_cifar10/train --quiet
!zip -r /kaggle/working/augmented_cifar10/test.zip   /kaggle/working/augmented_cifar10/test --quiet

# download
from google.colab import files
files.download("/kaggle/working/augmented_cifar10/train.zip")
files.download("/kaggle/working/augmented_cifar10/test.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

def count_images_per_class(directory):
    class_counts = {}
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            count = 0
            for _, _, files in os.walk(class_path):
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                        count += 1
            class_counts[class_name] = count
    return class_counts

print(f"--- Image counts per class in augmented training directory ({AUGMENTED_TRAIN_DIR}) ---")
train_class_counts = count_images_per_class(AUGMENTED_TRAIN_DIR)
for class_name, count in train_class_counts.items():
    print(f"Class {class_name}: {count} images")

print(f"\n--- Image counts per class in augmented testing directory ({AUGMENTED_TEST_DIR}) ---")
test_class_counts = count_images_per_class(AUGMENTED_TEST_DIR)
for class_name, count in test_class_counts.items():
    print(f"Class {class_name}: {count} images")


--- Image counts per class in augmented training directory (/kaggle/working/augmented_cifar10/train) ---
Class 0: 15000 images
Class 1: 15000 images
Class 2: 15000 images
Class 3: 15000 images
Class 4: 15000 images
Class 5: 15000 images
Class 6: 15000 images
Class 7: 15000 images
Class 8: 15000 images
Class 9: 15000 images

--- Image counts per class in augmented testing directory (/kaggle/working/augmented_cifar10/test) ---
Class 0: 3000 images
Class 1: 3000 images
Class 2: 3000 images
Class 3: 3000 images
Class 4: 3000 images
Class 5: 3000 images
Class 6: 3000 images
Class 7: 3000 images
Class 8: 3000 images
Class 9: 3000 images


# Load Data

In [ ]:
DATA_DIR = "/kaggle/working/cifar10_files/"

# Global access and constants
IMG_SIZE = (32, 32)
BATCH_SIZE = 128
shift_val = 1.0 / 28
SEED = 42
epochs = 250
num_classes=10
# Standard
weight_decay=1e-4  # chnaged from 5e-4
initial_learning_rate = 0.001 # changed from 1e-2
momentum = 0.9

train_ds, val_ds, test_ds, preprocessing_model = None, None, None, None

def get_keras_dataset():
    print("--- Loading CIFAR-10 Digits from local PNG folders ---")

    # Global access and constants
    global IMG_SIZE
    global BATCH_SIZE
    global SEED

    # 1. Load TRAIN dataset
    train_ds = tf.keras.utils.image_dataset_from_directory(
        f"{DATA_DIR}/train",
        validation_split=0.2,
        subset="training",
        seed=SEED,
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        f"{DATA_DIR}/train",
        validation_split=0.2,
        subset="validation",
        seed=SEED,
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    # 2. Load TEST dataset
    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory=f"{DATA_DIR}/test",
        labels="inferred",
        label_mode="categorical",
        color_mode="rgb",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    class_names = train_ds.class_names
    print("Class names:", class_names)

    # 4. Normalization layer (NO augmentation)
    norm_layer = layers.Normalization(axis=None)
    # Adapt on a mapping to ensure the stats are based on [0, 1] range
    norm_layer.adapt(
        train_ds.map(lambda x, y: x / 255.0).take(500)
    )

    # Fixed 1-pixel translation layer
    translation_layer = layers.RandomTranslation(
        height_factor=(shift_val, shift_val),
        width_factor=(shift_val, shift_val),  # Fixed at +1 pixel horizontally
        fill_mode='constant',
        fill_value=0.0,
        seed=SEED
    )

    preprocessing_model = models.Sequential([
        layers.Rescaling(1./255),
        norm_layer,
        layers.RandomRotation(0.15),
        translation_layer,
        layers.RandomZoom(0.02)
    ], name="preprocessing_head")

    # 5. Final pipeline
    def finalize(ds, shuffle=False):
        if shuffle:
            ds = ds.shuffle(10000, seed=SEED)
        return ds.prefetch(tf.data.AUTOTUNE)

    train_ds = finalize(train_ds, shuffle=True)
    val_ds = finalize(val_ds)
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

    return train_ds, val_ds, test_ds, preprocessing_model